# 산업재해 데이터 전처리

원본은 수정하지 않고 `raw-data` CSV를 검사한 뒤 분석용 복사본에서 Wide → Long 변환을 준비한다. 현재 파일은 SIF 사고사례 원본이 아닌 산업중분류별·규모별 사고사망자수 집계 데이터다.

In [11]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
RAW_PATH = Path("raw-data/한국산업안전보건공단_산업중분류별 규모별 사고사망자수_20251231.csv")
df = pd.read_csv(RAW_PATH, encoding='cp949')

In [12]:
# df.info()
# df.isna().sum()
# df.isna().mean()
df.dtypes


대업종                   str
구분                    str
5인 미만 근로자수          int64
5인 - 9인 근로자수        int64
10인 - 19인 근로자수      int64
20인 - 29인 근로자수      int64
30인 - 49인 근로자수      int64
50인 - 99인 근로자수      int64
100인 - 299인 근로자수    int64
300인 - 499인 근로자수    int64
500인 - 999인 근로자수    int64
1000인 이상 근로자수       int64
dtype: object

In [ ]:
inspection = pd.DataFrame({
    "dtype": df.dtypes,
    "missing_count": df.isna().sum(),
    "missing_rate": df.isna().mean().round(4),
    "unique_count": df.nunique(dropna=False)
})

display(inspection)
print("완전 중복 행 수:", df.duplicated().sum())

,dtype,missing_count,missing_rate,unique_count
대업종,str,0,0.0,10
구분,str,0,0.0,30
5인 미만 근로자수,int64,0,0.0,13
5인 - 9인 근로자수,int64,0,0.0,9
10인 - 19인 근로자수,int64,0,0.0,9
20인 - 29인 근로자수,int64,0,0.0,8
30인 - 49인 근로자수,int64,0,0.0,9
50인 - 99인 근로자수,int64,0,0.0,8
100인 - 299인 근로자수,int64,0,0.0,7
300인 - 499인 근로자수,int64,0,0.0,4


완전 중복 행 수: 0


In [4]:
work_df = df.rename(columns={
    "대업종": "대업종_raw",
    "구분": "구분_raw"
}).copy()

for col in ["대업종", "구분"]:
    work_df[col] = (
        work_df[f"{col}_raw"]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

size_columns = [col for col in work_df if "근로자수" in col]

long_df = work_df.melt(
    id_vars=["대업종_raw", "구분_raw", "대업종", "구분"],
    value_vars=size_columns,
    var_name="사업장규모",
    value_name="사고사망자수"
)

long_df["사고사망자수"] = pd.to_numeric(
    long_df["사고사망자수"], errors="coerce"
)

display(long_df.head())

,대업종_raw,구분_raw,대업종,구분,사업장규모,사고사망자수
0,금융및보험업,금융및보험업,금융및보험업,금융및보험업,5인 미만 근로자수,0
1,광 업,석탄광업및채석업,광 업,석탄광업및채석업,5인 미만 근로자수,0
2,광 업,석회석·금속·비금속광업및기타광업,광 업,석회석·금속·비금속광업및기타광업,5인 미만 근로자수,0
3,제조업,식료품제조업,제조업,식료품제조업,5인 미만 근로자수,5
4,제조업,섬유및섬유제품제조업,제조업,섬유및섬유제품제조업,5인 미만 근로자수,2


In [ ]:
expected_rows = len(df) * len(size_columns)

assert len(long_df) == expected_rows
assert long_df["사고사망자수"].notna().all()
assert (long_df["사고사망자수"] >= 0).all()

변환 전: (30, 12)
변환 후: (300, 6)
사업장 규모 수: 10
후보 Key 중복 수: 0


사업장규모
1000인 이상 근로자수        34
100인 - 299인 근로자수     74
10인 - 19인 근로자수      113
20인 - 29인 근로자수       49
300인 - 499인 근로자수     12
30인 - 49인 근로자수       63
500인 - 999인 근로자수     19
50인 - 99인 근로자수       47
5인 - 9인 근로자수        107
5인 미만 근로자수          354
Name: 사고사망자수, dtype: int64

In [6]:
print("변환 전:", df.shape)
print("변환 후:", long_df.shape)
print("사업장 규모 수:", long_df["사업장규모"].nunique())
print("후보 Key 중복 수:", long_df[["대업종", "구분", "사업장규모"]].duplicated().sum())
display(long_df.groupby("사업장규모")["사고사망자수"].sum())

변환 전: (30, 12)
변환 후: (300, 6)
사업장 규모 수: 10
후보 Key 중복 수: 0


사업장규모
1000인 이상 근로자수        34
100인 - 299인 근로자수     74
10인 - 19인 근로자수      113
20인 - 29인 근로자수       49
300인 - 499인 근로자수     12
30인 - 49인 근로자수       63
500인 - 999인 근로자수     19
50인 - 99인 근로자수       47
5인 - 9인 근로자수        107
5인 미만 근로자수          354
Name: 사고사망자수, dtype: int64